In [1]:
## Author: Tanay Roy
## Date: 27 Jan 2026

# Preamble

In [2]:
from src.parameter_finder import pulse_parameter_finder, add_buffer_levels, \
                                 construct_U_realized, display_params
import numpy as np
from numpy import pi, sqrt, exp
# np.set_printoptions(precision=5)

In [3]:
def overlap_unitary_fidelity(Uid, U):
    d = Uid.shape[0]
    try:
        Uid, U = Uid.full(), U.full()
    except:
        pass
    return (np.abs(np.trace(Uid.conj().T @ U)))/d

def overlap_state_fidelity(psi_id, psi):
    try:
        fid = fidelity(psi_id, psi)
    except:
        fid = np.abs(psi_id.conj().T @ psi)[0][0]
    return fid**2

def chop(x, delta=1e-4):
    '''Replace numbers<delta with 0'''
    x = np.array(x)
    if x.dtype == 'int32' or x.dtype == 'float64':
        x[abs(x) < delta] = 0
    if x.dtype == 'complex128':
        x.real[abs(x.real) < delta] = 0
        x.imag[abs(x.imag) < delta] = 0
    else: print('Operation incompatible')
    return x
    
def hadamard(d):
    '''Qudit Hadamard gate'''
    w = exp(2j*pi/d)
    mat = np.ones([d,d], dtype=complex)
    for ii in np.arange(1,d):
        for jj in np.arange(1,d):
            mat[ii,jj] = w**(ii*jj)
    return mat/sqrt(d)

# Qutrit Hadamard gate (d=3)

In [12]:
# Target gate
qudit_gate = hadamard(3)

d_cut = 12 # cutoff dimension = qudit_dim + buffer_levels
target_cavity_gate = add_buffer_levels(qudit_gate, d_cut) # adding buffer levels

print("qudit_gate:")
print(qudit_gate)

qudit_gate:
[[ 0.70710678+0.00000000e+00j  0.70710678+0.00000000e+00j]
 [ 0.70710678+0.00000000e+00j -0.70710678+8.65956056e-17j]]


## Run optimizer

In [13]:
d_snap, n_snap = 3, 2 # snap action dimension, no. of (multi-parameter) snap pulses

use_guess = True # If True, set displacement & phase values below in 'guess'
if use_guess:
    guess = np.concatenate(([-0.5,0.8,0.5], (np.pi) * np.random.randint(2, size=n_snap*d_snap))) 
    # displacement array length should be (n_snap + 1) if use_guess = 1
    output = pulse_parameter_finder(target_cavity_gate, n_snap, d_cut, n_levels=d_snap, max_runs=50,\
                                    err_th=0.01, d_fid=10, initial_guess = guess)
else:
    output = pulse_parameter_finder(target_cavity_gate, n_snap, d_cut, n_levels=d_snap, max_runs=50,\
                                    err_th=0.02, d_fid=10, initial_guess = None)
    # Optimizer stops if infidelity < err_th
    # d_cut >= d_fid >= qudit dimension; d_fid is used to compute unitary fidelity
display_params(output[1], n_snap)
print(f"Fidelity, infidelity (d=d_fid): {round(1-output[3],6)} , {round(output[3],6)}")

Current itr#: 0
Displacements: [-0.025033   -0.34194108  0.36571769]
SNAP degrees:
[[ 1.814e+02  1.791e+02  3.000e-01]
 [-1.820e+02  1.000e-01 -1.000e-01]]
Fidelity, infidelity (d=d_fid): 0.996437 , 0.003563


In [14]:
# Use 1 extra snap
d_snap, n_snap = 3, 3 # snap action dimension, no. of snap pulses

use_guess = 1
if use_guess:
    guess = np.concatenate(([0.5,-0.3,0.5,-0.3], (np.pi) * np.random.randint(2, size=n_snap*d_snap))) 
    # displacement array length should be (n_snap + 1) if use_guess = 1
    output = pulse_parameter_finder(target_cavity_gate, n_snap, d_cut, n_levels=d_snap, max_runs=50,\
                                    err_th=0.01, d_fid=6, initial_guess = guess)
else:
    output = pulse_parameter_finder(target_cavity_gate, n_snap, d_cut, n_levels=d_snap, max_runs=50,\
                                    err_th=0.02, d_fid=10, initial_guess = None)
display_params(output[1], n_snap)
print(f"Fidelity, infidelity (d=d_fid): {round(1-output[3],6)} , {round(output[3],6)}")

Current itr#: 0


Exception: Optimiser did not converge successfully.

## Verify decomposition

In [11]:
# Check fidelity within qudit space
d_qudit, d_cut_new = 3, d_cut + 10
mat = construct_U_realized(output[1], d_cut_new, n_snap)
print(f"Fidelity (cutoff d = {d_cut_new}): {overlap_unitary_fidelity(mat, add_buffer_levels(qudit_gate, d_cut_new)):.6f}")
print(f"Fidelity (qudit d = {d_qudit}): {overlap_unitary_fidelity(mat[:d_qudit,:d_qudit], add_buffer_levels(qudit_gate, d_qudit)):.6f}")

ValueError: cannot reshape array of size 5 into shape (3,1)

In [15]:
# Check argument of the complex unitary
np.rad2deg(np.angle(mat[:d_qudit,:d_qudit]))

NameError: name 'mat' is not defined

# Permutation gate (d=4)

In [16]:
qudit_gate = np.array([[ 0.,  1.,  0.,  0.], # target gate
                       [ 1.,  0.,  0.,  0.],
                       [ 0.,  0.,  0.,  1.],
                       [ 0.,  0.,  1.,  0.]])

d_cut = 12 # cutoff dimension
target_cavity_gate = add_buffer_levels(qudit_gate, d_cut) # adding buffer levels

## Run optimizer

In [17]:
d_snap, n_snap = 3, 4 # snap action dimension, no. of snap pulses

use_guess = 0
if use_guess:
    guess = np.concatenate(([0.2,-0.4,0.5,-0.4,0.2], (np.pi) * np.random.randint(2, size=n_snap*d_snap))) 
    # displacement array length should be (n_snap + 1) if use_guess = 1
    output = pulse_parameter_finder(target_cavity_gate, n_snap, d_cut, n_levels=d_snap, max_runs=50,\
                                    err_th=0.1, d_fid=10, initial_guess = guess)
else:
    output = pulse_parameter_finder(target_cavity_gate, n_snap, d_cut, n_levels=d_snap, max_runs=50,\
                                    err_th=0.01, d_fid=10, initial_guess = None)

display_params(output[1], n_snap)
print(f"Fidelity, infidelity (d=d_fid): {round(1-output[3],6)} , {round(output[3],6)}")

Current itr#: 0
Current itr#: 10
Current itr#: 20
Displacements: [ 0.20294737 -0.39326127  0.26834318  0.28591735 -0.36058545]
SNAP degrees:
[[331.4 164.1 184.4]
 [-48.7 122.4 158.7]
 [ 48.8  37.9 181.9]
 [202.6  44.4  16.9]]
Fidelity, infidelity (d=d_fid): 0.990639 , 0.009361


## Verify decomposition

In [18]:
# Check fidelity within qudit space
d_qudit, d_cut_new = 4, d_cut + 10
mat = construct_U_realized(output[1], d_cut_new, n_snap)
print(f"Fidelity (cutoff d = {d_cut_new}): {overlap_unitary_fidelity(mat, add_buffer_levels(qudit_gate, d_cut_new)):.6f}")
print(f"Fidelity (qudit d = {d_qudit}): {overlap_unitary_fidelity(mat[:d_qudit,:d_qudit], add_buffer_levels(qudit_gate, d_qudit)):.6f}")

Fidelity (cutoff d = 22): 0.995644
Fidelity (qudit d = 4): 0.978670
